In [ ]:
import os, torch
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:128'
torch.cuda.empty_cache()
import os, sys
os.environ["TORCH_NVML_DISABLED"] = "1"
os.chdir("/scratch/jq2uw/MME/instruct_vlm_edit")

repo_root = "/scratch/jq2uw/MME/instruct_vlm_edit"
if repo_root not in sys.path:
    sys.path.append(repo_root)

from revlm.config_utils import *
from revlm.dataset import *
from revlm.models import *
import argparse

In [ ]:
cfg_path = os.path.join(repo_root, "revlm", "config", "config.yaml")

# Simulate CLI overrides
args = argparse.Namespace(
    config=cfg_path,
    editor="ft",  # should become config.editor._name
    inner_params=["transformer.h.0.mlp.c_fc.weight"],  # -> config.model.inner_params
    dataset_name="aokvqa",#"fvqa",  # -> config.experiment.dataset_name
    model_name="llava",
)

config = configure_args(args, config_path=cfg_path)
# config.experiment.streaming = True
config

In [ ]:
vlm = VQAModel(config)

In [ ]:
# ds = get_dataset(config, split="train")
# print(f"Edit dataset size: {len(ds)}")
# ds.set_dataloader(task="mci", shuffle_choices=True, seed=333, batch_size=64)

# # ds.data = random.sample(ds.data, 10)
# # ds.task_generate(vlm)
# # ds.data[0]

In [ ]:
edit_dataset = VQADataset(config)
edit_dataset.data = random.sample(edit_dataset.data, 10)
# print(f"Edit dataset size: {len(edit_dataset)}")
edit_dataset.set_dataloader()
edit_dataset.task_generate(vlm)

mc

In [ ]:
# edit_dataset.task_generate(batch, vlm)
edit_dataset.data[0]
edit_dataset.task_engineer.eval(edit_dataset)

In [ ]:

# g_labels = [g['label'] for g in batch['golds']]
# label_words = [g['choices']['ls'] for g in batch['golds']]

# s = vlm.score_choices(images, prompts, label_words)
# for ex in edit_dataset.data:
#     s = vlm.score_choices_single(ex['image'], ex['prompt'], ex['gold']['choices']['ls'])
#     break
# s

In [ ]:
edit_dataset.set_dataloader(task="mci", shuffle_choices=True, seed=333, batch_size=64)
for batch in edit_dataset.loader:
    images = batch['images']
    prompts = batch['prompts']
    golds = batch['golds']
    break

In [ ]:
g_letters = [g['label_letter'] for g in batch['golds']]
g_labels = [g['label'] for g in batch['golds']]

use_letter = False
letter_lists = [[c[0] for c in g['choices']['ls']] for g in batch['golds']]
text_lists   = [[c[1] for c in g['choices']['ls']] for g in batch['golds']]
label_words = letter_lists if use_letter else text_lists

s = vlm.score_choices(images, prompts, text_lists)
c = vlm.score_choices(images, prompts, letter_lists)

In [ ]:
correct = 0
for i in range(len(s)):
    d = s[i]
    best_label = max(d, key=lambda k: d[k]['prob'])
    pred_text = (best_label[1] if isinstance(best_label, (tuple, list)) else best_label)
    gold_text = str(g_letters[i]) if use_letter else str(g_labels[i])
    if str(pred_text).strip().lower() == gold_text.strip().lower():
        correct += 1
acc = correct / len(s) if s else 0.0
print(f"Batch accuracy: {acc:.4f} ({correct}/{len(s)})")


In [ ]:
model = vlm
for ex in edit_dataset.data:
    ex['pred'] = {}
    # score-based generation
    label_texts = [choice for _, choice in ex['gold']['choices']['ls']]
    label_letters = [ltr for ltr, _ in ex['gold']['choices']['ls']]
    ex['pred']['label_scores'] = model.score_choices_single(ex['image'], ex['prompt'], label_texts)
    ex['pred']['letter_scores'] = model.score_choices_single(ex['image'], ex['prompt'], label_letters)
    ex['pred']['label_maxprob'] = max(ex['pred']['label_scores'], key=lambda k: ex['pred']['label_scores'][k]['prob'])
    ex['pred']['letter_maxprob'] = max(ex['pred']['letter_scores'], key=lambda k: ex['pred']['letter_scores'][k]['prob'])
        
    break

edit_dataset.data[0]

In [ ]:
max(s, key=lambda k: s[k]['prob'])

In [ ]:
correct = 0
for i in range(len(s)):
    d = s[i]
    best_label = max(d, key=lambda k: d[k]['prob'])
    pred_text = (best_label[1] if isinstance(best_label, (tuple, list)) else best_label)
    gold_text = str(g_letters[i]) if use_letter else str(g_labels[i])
    if str(pred_text).strip().lower() == gold_text.strip().lower():
        correct += 1
acc = correct / len(s) if s else 0.0
print(f"Batch accuracy: {acc:.4f} ({correct}/{len(s)})")


In [ ]:
for i in range(len(s)):
    print(str(g_letters[i]) + " " + str(g_labels[i]))
    print(label_words[i])
    d = s[i]
    best_label = max(d, key=lambda k: d[k]['prob'])
    best_prob = d[best_label]['prob']
    print(best_label, best_prob)

In [ ]:
from revlm.metrics import QA_metrics_text, QA_metrics_loss, QA_metrics_nli_bi

# Load test split and prepare loader
test_dataset = get_dataset(config, split="train")
import random
subsample = 90
if len(test_dataset) > subsample:
    test_dataset.data = random.sample(test_dataset.data, subsample)
test_dataset.set_dataloader(
    task="qa",
    with_rationale=False,
    shuffle_choices=False,
    batch_size=32,
)

# Run metrics
res_text = QA_metrics_text(vlm, test_dataset)
res_loss = QA_metrics_loss(vlm, test_dataset)
res_nli = QA_metrics_nli_bi(vlm, test_dataset)



print("QA Text parse:", res_text)
print("QA Log-loss:", res_loss)
print("QA NLI Bi:", res_nli)

In [ ]:
# Evaluate MCQ on FVQA test split
from revlm.metrics.mcq import MCQ_metrics_text, MCQ_metrics_score, MCQ_metrics_classifier
test_dataset.set_dataloader(
    task="mcq",
    with_rationale=False,
    shuffle_choices=False,
    batch_size=32,
)

# Run metrics
res_text = MCQ_metrics_text(vlm, test_dataset)
res_score = MCQ_metrics_score(vlm, test_dataset, score_by_letter=True)
res_score_option = MCQ_metrics_score(vlm, test_dataset, score_by_letter=False)
res_cls = MCQ_metrics_classifier(vlm, test_dataset)



print("MCQ Text parse:", res_text)
print("MCQ Log-loss:", res_score)
print("MCQ Log-loss (option):", res_score_option)
print("MCQ Classifier:", res_cls)


In [ ]:
test_dataset.set_dataloader(
    task="mcq",
    with_rationale=False,
    shuffle_choices=True,
    batch_size=32,
)

# Run metrics
res_text = MCQ_metrics_text(vlm, test_dataset)
res_score = MCQ_metrics_score(vlm, test_dataset)
res_cls = MCQ_metrics_classifier(vlm, test_dataset)

print("MCQ Text parse:", res_text)
print("MCQ Log-loss:", res_score)
print("MCQ Classifier:", res_cls)


In [ ]:
edit_dataset = get_dataset(config)
print(f"Edit dataset size: {len(edit_dataset)}")
edit_dataset.shuffle_choices()
edit_dataset.data[0]


choices = []
imgs = []
questions = []
for i in range(20):
    ex = edit_dataset.data[i]
    choices.append(ex["choices"])
    imgs.append(ex["image"]) # read image fomr ex["image_path"]
    questions.append("Choose A/B/C/D based on the image."+ex["question"] + ex['choices'])

In [ ]:

with torch.no_grad():
    ans = vlm.generate(images=imgs, prompts=questions, max_new_tokens=100)

ans